# Pipeline de Ingestão - Camada Bronze
## Projeto Câmara Brasil

Este notebook realiza a ingestão de dados da API de Dados Abertos da Câmara dos Deputados e persiste os arquivos brutos em formato Delta na camada Bronze.

**Fontes de dados:**
- Frentes Parlamentares
- Deputados
- Eventos
- Votações
- Despesas (CEAP)
- CPIs

In [0]:
import json
import math
import os
from typing import Dict, Iterable, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import time

import requests
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import lit

# Configurações globais
BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"
CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = "bronze_camara_"
MAX_WORKERS = 20  # Threads paralelas para chamadas API (aumentado para melhor performance)
RETRY_ATTEMPTS = 3  # Tentativas de retry em caso de erro temporário

print(f"✓ Configuração carregada")
print(f"  BASE_URL: {BASE_URL}")
print(f"  Catálogo UC: {CATALOG}.{SCHEMA}")
print(f"  Prefixo das tabelas: {TABLE_PREFIX}")
print(f"  Paralelização: {MAX_WORKERS} workers")
print(f"  Retry: {RETRY_ATTEMPTS} tentativas")

In [0]:
def get_api_data(endpoint: str, params: Optional[Dict[str, str]] = None) -> List[Dict]:
    """Realiza chamadas HTTP à API com paginação automática e retry."""
    if params is None:
        params = {}
    
    url = f"{BASE_URL}/{endpoint}"
    results: List[Dict] = []
    
    while url:
        # Retry logic para chamadas API
        for attempt in range(RETRY_ATTEMPTS):
            try:
                response = requests.get(url, params=params, timeout=30)
                response.raise_for_status()
                payload = response.json()
                results.extend(payload.get("dados", []))
                
                # Procura link para próxima página
                links = payload.get("links", [])
                next_link = None
                for link in links:
                    if link.get("rel") == "next":
                        next_link = link.get("href")
                        break
                url = next_link
                params = None
                break  # Sucesso, sai do retry loop
                
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < RETRY_ATTEMPTS - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                    continue
                else:
                    raise  # Última tentativa falhou
            except requests.exceptions.HTTPError as e:
                # Não faz retry para erros 4xx (cliente)
                if 400 <= e.response.status_code < 500:
                    raise
                # Retry para erros 5xx (servidor)
                if attempt < RETRY_ATTEMPTS - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    raise
    
    return results

def normalize_complex_fields(records: List[Dict]) -> List[Dict]:
    """Converte campos complexos (dict/list) para JSON strings de forma eficiente."""
    for rec in records:
        for key in list(rec.keys()):
            value = rec[key]
            if isinstance(value, (dict, list)):
                rec[key] = json.dumps(value)
    return records

def fetch_with_id(fetch_func, item_id):
    """Wrapper para chamadas API com ID que retorna (id, resultado)."""
    try:
        result = fetch_func(item_id)
        return (item_id, result, None)
    except Exception as e:
        return (item_id, None, str(e))

def save_bronze(df: DataFrame, table_name: str, spark: SparkSession) -> None:
    """Grava DataFrame no formato Delta na camada Bronze."""
    full_table_name = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}{table_name}"
    (df
     .write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(full_table_name))
    print(f"✓ Tabela '{full_table_name}' gravada com sucesso")

print("✓ Funções auxiliares carregadas")

In [0]:
def ingest_frentes(spark: SparkSession) -> DataFrame:
    """Ingesta lista de frentes parlamentares."""
    print("Ingerindo frentes parlamentares...")
    records = get_api_data("frentes")
    records = normalize_complex_fields(records)
    
    if not records:
        print("  Nenhuma frente encontrada")
        return spark.createDataFrame([], schema="id LONG")
    
    df = spark.createDataFrame(records)
    save_bronze(df, "frentes", spark)
    print(f"  Total: {df.count()} frentes")
    return df

def ingest_frentes_membros(spark: SparkSession) -> DataFrame:
    """Ingesta membros de cada frente com paralelização."""
    print("Ingerindo membros das frentes...")
    frentes_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}frentes")
    frentes_ids = [row.id for row in frentes_df.select("id").distinct().collect()]
    
    print(f"  Processando {len(frentes_ids)} frentes em paralelo com {MAX_WORKERS} workers...")
    
    def fetch_members(fid):
        """Busca membros de uma frente."""
        try:
            members = get_api_data(f"frentes/{fid}/membros")
            for m in members:
                m["idFrente"] = fid
            return members
        except Exception:
            return []
    
    all_members: List[Dict] = []
    errors = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {executor.submit(fetch_members, fid): fid for fid in frentes_ids}
        
        for future in as_completed(future_to_id):
            fid = future_to_id[future]
            try:
                members = future.result()
                if members:  # Só adiciona se tiver membros
                    all_members.extend(members)
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  Aviso: Erro ao processar membros da frente {fid}: {e}")
    
    if errors > 0:
        print(f"  Total de erros: {errors} frentes")
    
    if not all_members:
        print("  Nenhum membro encontrado")
        from pyspark.sql.types import StructType, StructField, LongType, StringType
        schema = StructType([
            StructField("idFrente", LongType(), True),
            StructField("nome", StringType(), True)
        ])
        return spark.createDataFrame([], schema=schema)
    
    # Normaliza campos complexos
    all_members = normalize_complex_fields(all_members)
    
    # Identifica campos comuns em todos os registros
    if all_members:
        # Pega campos do primeiro registro
        common_keys = set(all_members[0].keys())
        # Intersecta com campos dos outros registros
        for m in all_members[1:20]:  # Verifica primeiros 20 registros
            common_keys = common_keys.intersection(set(m.keys()))
        
        # Garante que pelo menos idFrente e id existam
        if "idFrente" not in common_keys or "id" not in common_keys:
            print(f"  Aviso: Campos inconsistentes detectados. Usando apenas campos básicos.")
            basic_members = []
            for m in all_members:
                basic = {"idFrente": m.get("idFrente"), "id": m.get("id")}
                if m.get("nome"):
                    basic["nome"] = m.get("nome")
                if basic["idFrente"] and basic["id"]:
                    basic_members.append(basic)
            all_members = basic_members
    
    # Tenta criar DataFrame
    try:
        df = spark.createDataFrame(all_members)
    except Exception as e:
        print(f"  Erro ao criar DataFrame: {e}")
        print(f"  Usando apenas campos básicos como fallback...")
        # Extrai apenas campos básicos que sempre existem
        basic_members = [{"idFrente": m.get("idFrente"), "id": m.get("id"), "nome": str(m.get("nome", ""))[:500]} 
                         for m in all_members if m.get("idFrente") and m.get("id")]
        if not basic_members:
            from pyspark.sql.types import StructType, StructField, LongType, StringType
            schema = StructType([
                StructField("idFrente", LongType(), True),
                StructField("id", LongType(), True),
                StructField("nome", StringType(), True)
            ])
            return spark.createDataFrame([], schema=schema)
        df = spark.createDataFrame(basic_members)
    
    save_bronze(df, "frentes_membros", spark)
    print(f"  Total: {df.count()} membros")
    return df

print("✓ Funções de frentes carregadas")

In [0]:
def ingest_deputados(spark: SparkSession) -> DataFrame:
    """Ingesta lista de deputados em exercício."""
    print("Ingerindo deputados...")
    records = get_api_data("deputados")
    records = normalize_complex_fields(records)
    
    if not records:
        print("  Nenhum deputado encontrado")
        return spark.createDataFrame([], schema="id LONG")
    
    df = spark.createDataFrame(records)
    save_bronze(df, "deputados", spark)
    print(f"  Total: {df.count()} deputados")
    return df

print("✓ Função de deputados carregada")

In [0]:
def ingest_eventos(spark: SparkSession, ano: Optional[int] = None) -> DataFrame:
    """Ingesta eventos legislativos."""
    print(f"Ingerindo eventos{' de ' + str(ano) if ano else ''}...")
    params = {}
    if ano:
        params["dataInicio"] = f"{ano}-01-01"
        params["dataFim"] = f"{ano}-12-31"
    records = get_api_data("eventos", params=params)
    records = normalize_complex_fields(records)
    
    if not records:
        print("  Nenhum evento encontrado")
        return spark.createDataFrame([], schema="id LONG")
    
    df = spark.createDataFrame(records)
    save_bronze(df, f"eventos{'_'+str(ano) if ano else ''}", spark)
    print(f"  Total: {df.count()} eventos")
    return df

print("✓ Função de eventos carregada")

In [0]:
def ingest_votacoes(spark: SparkSession, data_inicio: Optional[str] = None, data_fim: Optional[str] = None) -> DataFrame:
    """Ingesta votações do período especificado."""
    print("Ingerindo votações...")
    params = {}
    if data_inicio:
        params["dataInicio"] = data_inicio
    if data_fim:
        params["dataFim"] = data_fim
    records = get_api_data("votacoes", params=params)
    records = normalize_complex_fields(records)
    
    if not records:
        print("  Nenhuma votação encontrada")
        return spark.createDataFrame([], schema="id LONG")
    
    df = spark.createDataFrame(records)
    save_bronze(df, "votacoes", spark)
    print(f"  Total: {df.count()} votações")
    return df

def ingest_votos_deputados(spark: SparkSession, votacao_ids: Optional[Iterable[int]] = None) -> DataFrame:
    """Ingesta votos por deputado com paralelização."""
    print("Ingerindo votos dos deputados...")
    if votacao_ids is None:
        vot_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}votacoes")
        votacao_ids = [row.id for row in vot_df.select("id").distinct().collect()]
    
    print(f"  Processando {len(votacao_ids)} votações em paralelo com {MAX_WORKERS} workers...")
    
    def fetch_votes(vid):
        """Busca votos de uma votação."""
        votes = get_api_data(f"votacoes/{vid}/votos")
        for v in votes:
            v["idVotacao"] = vid
        return votes
    
    all_votes: List[Dict] = []
    errors = 0
    not_found = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {executor.submit(fetch_votes, vid): vid for vid in votacao_ids}
        
        for future in as_completed(future_to_id):
            vid = future_to_id[future]
            try:
                votes = future.result()
                all_votes.extend(votes)
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 404:
                    not_found += 1
                else:
                    errors += 1
                    if errors <= 5:
                        print(f"  Aviso: Erro HTTP {e.response.status_code} ao obter votos da votação {vid}")
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  Aviso: Erro ao obter votos da votação {vid}: {e}")
    
    if not_found > 0:
        print(f"  {not_found} votações sem votos disponíveis (404)")
    if errors > 0:
        print(f"  Total de erros: {errors} votações")
    
    if not all_votes:
        print("  Nenhum voto encontrado")
        return spark.createDataFrame([], schema="idVotacao LONG, deputado STRING")
    
    all_votes = normalize_complex_fields(all_votes)
    df = spark.createDataFrame(all_votes)
    save_bronze(df, "votos_deputados", spark)
    print(f"  Total: {df.count()} votos")
    return df

print("✓ Funções de votações carregadas")

In [0]:
def ingest_despesas_deputados(spark: SparkSession, deputado_ids: Optional[Iterable[int]] = None) -> DataFrame:
    """Ingesta despesas por deputado com paralelização."""
    print("Ingerindo despesas dos deputados...")
    if deputado_ids is None:
        dep_df = spark.read.table(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}deputados")
        deputado_ids = [row.id for row in dep_df.select("id").distinct().collect()]
    
    print(f"  Processando despesas de {len(deputado_ids)} deputados em paralelo com {MAX_WORKERS} workers...")
    print(f"  Aviso: Este processo pode demorar alguns minutos devido ao volume de dados...")
    
    def fetch_expenses(dep_id):
        """Busca todas as despesas de um deputado (paginadas)."""
        expenses = []
        page = 1
        while True:
            params = {"pagina": page, "itens": 100}
            try:
                page_expenses = get_api_data(f"deputados/{dep_id}/despesas", params=params)
                if not page_expenses:
                    break
                for e in page_expenses:
                    e["idDeputado"] = dep_id
                expenses.extend(page_expenses)
                page += 1
            except Exception:
                break
        return expenses
    
    all_expenses: List[Dict] = []
    errors = 0
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {executor.submit(fetch_expenses, dep_id): dep_id for dep_id in deputado_ids}
        
        completed = 0
        for future in as_completed(future_to_id):
            dep_id = future_to_id[future]
            completed += 1
            if completed % 50 == 0:
                print(f"  Progresso: {completed}/{len(deputado_ids)} deputados processados")
            try:
                expenses = future.result()
                all_expenses.extend(expenses)
            except Exception as e:
                errors += 1
                if errors <= 5:
                    print(f"  Aviso: Erro ao obter despesas do deputado {dep_id}: {e}")
    
    if errors > 0:
        print(f"  Total de erros: {errors} deputados")
    
    if not all_expenses:
        print("  Nenhuma despesa encontrada")
        return spark.createDataFrame([], schema="idDeputado LONG")
    
    all_expenses = normalize_complex_fields(all_expenses)
    df = spark.createDataFrame(all_expenses)
    save_bronze(df, "despesas_deputados", spark)
    print(f"  Total: {df.count()} despesas")
    return df

def ingest_cpis(spark: SparkSession) -> DataFrame:
    """Ingesta dados de CPIs."""
    print("Ingerindo CPIs...")
    params = {"siglaTipo": "CPIPC"}
    records = get_api_data("proposicoes", params=params)
    records = normalize_complex_fields(records)
    
    if not records:
        print("  Nenhuma CPI encontrada")
        return spark.createDataFrame([], schema="id LONG")
    
    df = spark.createDataFrame(records)
    save_bronze(df, "cpis", spark)
    print(f"  Total: {df.count()} CPIs")
    return df

print("✓ Funções de despesas e CPIs carregadas")

In [0]:
def ingest_all(spark: SparkSession, ano: Optional[int] = None, data_inicio: Optional[str] = None, data_fim: Optional[str] = None) -> None:
    """Executa todas as ingestões disponíveis.
    
    Args:
        ano: Ano para filtrar eventos (ex: 2024)
        data_inicio: Data início para votações (formato: AAAA-MM-DD)
        data_fim: Data fim para votações (formato: AAAA-MM-DD)
    """
    print("="*60)
    print("INICIANDO PIPELINE DE INGESTÃO - CAMADA BRONZE")
    if ano:
        print(f"  Filtro de ano para eventos: {ano}")
    if data_inicio or data_fim:
        print(f"  Filtro de data para votações: {data_inicio} a {data_fim}")
    print("="*60)
    
    functions = [
        ("frentes", lambda s: ingest_frentes(s)),
        ("frentes_membros", lambda s: ingest_frentes_membros(s)),
        ("deputados", lambda s: ingest_deputados(s)),
        ("eventos", lambda s: ingest_eventos(s, ano=ano)),
        ("votacoes", lambda s: ingest_votacoes(s, data_inicio=data_inicio, data_fim=data_fim)),
        ("votos_deputados", lambda s: ingest_votos_deputados(s)),
        ("despesas_deputados", lambda s: ingest_despesas_deputados(s)),
        ("cpis", lambda s: ingest_cpis(s))
    ]
    
    failed = []
    for name, func in functions:
        try:
            func(spark)
        except Exception as e:
            error_msg = f"{name}: {type(e).__name__}: {str(e)[:100]}"
            print(f"✗ Erro em {name}: {type(e).__name__}")
            failed.append(error_msg)
    
    print("="*60)
    if not failed:
        print("✓ PIPELINE BRONZE CONCLUÍDO COM SUCESSO!")
    else:
        print(f"⚠ PIPELINE CONCLUÍDO COM {len(failed)} ERRO(S)")
        for err in failed:
            print(f"  - {err}")
    print("="*60)

# EXECUTAR PIPELINE COM PARÂMETROS
# Lê parâmetros dos widgets
try:
    ano_str = dbutils.widgets.get("ano")
    data_inicio_str = dbutils.widgets.get("data_inicio")
    data_fim_str = dbutils.widgets.get("data_fim")
    
    # Converte string para int se houver valor
    ano_param = int(ano_str) if ano_str and ano_str.strip() and ano_str != "None" else None
    data_inicio_param = data_inicio_str if data_inicio_str and data_inicio_str.strip() and data_inicio_str != "None" else None
    data_fim_param = data_fim_str if data_fim_str and data_fim_str.strip() and data_fim_str != "None" else None
except:
    # Fallback: sem filtros
    ano_param = None
    data_inicio_param = None
    data_fim_param = None

ingest_all(spark, ano=ano_param, data_inicio=data_inicio_param, data_fim=data_fim_param)